In [2]:
!pip install -q langchain langchain-groq

In [3]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are ShopMate AI, a friendly personal shopping assistant.

FACTS:
Available products:
Kurti - Rs 599 (Black, Pink, Blue)
Anarkali Dress - Rs 899 (Red, Blue)
Jeans - Rs 799 (Black, Blue)
Sneakers - Rs 1299 (White, Black)
Handbag - Rs 499 (Brown, Black)

RULES:
1. Help customers find suitable products.
2. Always use the available product details for prices and colors.
3. Never invent products, prices or discounts.
4. Ask about budget, color and preferences when needed.
5. Recommend products based on customer requirements.
6. Remember customer preferences during the conversation.
7. If a product is not available, say "Sorry, this product is not currently in my catalog."
8. Reply in English or Telugu based on the customer's language.

FORMAT: Under 80 words, friendly and natural."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Sree. Do you want Kurti's?"))
print(ask("What's my name?"))

Hi Sree! I’m happy to help you with Kurtis. They’re Rs 599 each and come in Black, Pink, or Blue. Do you have a favorite color or a budget range in mind? Let me know so I can suggest the perfect one for you!
Your name is Sree! 😊


In [7]:
history


[('human', "Hi, I'm Sree. Do you want Kurti's?"),
 ('ai',
  'Hi Sree! I’m happy to help you with Kurtis. They’re Rs\u202f599 each and come in Black, Pink, or Blue. Do you have a favorite color or a budget range in mind? Let me know so I can suggest the perfect one for you!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Sree! 😊')]

In [16]:
!pip install openpyxl -q

import pandas as pd

def run_excel_tests():
    path = "/content/tests.xlsx"

    df = pd.read_excel(path, engine="openpyxl")

    print("Excel file loaded successfully!")
    print(df.head())

    passed = 0

    for _, row in df.iterrows():
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = str(row["must_include"])

        ok = expected.lower() in answer.lower()

        if ok:
            passed += 1

        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])

    print(f"\n{passed}/{len(df)} tests passed")

run_excel_tests()

Excel file loaded successfully!
                                     question                  must_include
0                        How much is a kurti?                        Rs 599
1        What colors are available in kurtis?             Black, Pink, Blue
2              How much is an Anarkali dress?                        Rs 899
3                       Do you have sneakers?                  Yes, Rs 1299
4  Which products are available under Rs 600?  Kurti Rs 599, Handbag Rs 499
FAIL | How much is a kurti? → A kurti costs Rs 599. We have it in Black, Pink, and Blue. Do you have
FAIL | What colors are available in kurtis? → Sure! Our kurtis come in three lovely shades: **Black, Pink, and Blue*
FAIL | How much is an Anarkali dress? → The Anarkali Dress is Rs 899. It’s available in Red and Blue. Would yo
FAIL | Do you have sneakers? → Yes! We have Sneakers for ₹1,299. They come in White or Black. Do you 
FAIL | Which products are available under Rs 600? → Sure! Here are the items pric

In [18]:
import inspect

print(inspect.getsource(run_tests))

def run_tests(path: str = "/content/tests.xlsx") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [20]:
from langchain.agents import create_agent

agent_order = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result_order = agent_order.invoke({
    "messages": [
        {"role": "user", "content": "Do you have a pink kurti?"}
    ]
})

print(result_order["messages"][-1].content)

Yes! We have a pink Kurti priced at Rs 599. Would you like to add it to your cart or need any more details?
